# Project 1 — Advanced bank accounts (complete solution)

**Requirements:** Python 3.11 or newer; **dependencies:** Python standard library only. Run the cells top to bottom. This notebook is self-contained, independently executable, and includes a `unittest` suite. Python is **not** constrained to the original exercise's Python 3.6.

### Core requirements and deliberate decisions

- Accounts have an immutable, digit-only **string** account number (preserving leading zeros), validated names, preferred timezone, and a read-only nonnegative balance.
- Every accepted deposit, withdrawal, and interest credit generates a `D`, `W`, or `I` confirmation; insufficient funds generate `X` without changing the balance. Invalid monetary inputs raise rather than creating fake transactions.
- UTC timestamps and a lock-protected, **process-wide** increasing ID form `CODE-ACCOUNT-YYYYMMDDHHMMSS-ID`. UTC timestamps have **second** precision in the confirmation, so transaction IDs distinguish same-second events.
- Currency uses `Decimal` and cents, not floating-point accounting. Explicit inputs containing fractions of a cent are rejected; **calculated interest** rounds half-to-even to the nearest cent. A finite maximum balance prevents decimal-context precision loss for normal operations.
- The monthly interest rate is shared by **all** accounts. `0.005` is **0.5%**, producing **1,005.00** on a 1,000.00 balance; the exercise's **1,050.00** example requires `0.05`, or **5%**.
- `Account.parse_confirmation()` is a **static method**: it needs only the code and a desired timezone, not account instance state. Parsing is not authentication.

### Added functionality

An immutable ledger with before/after balances; searchable receipt verification against that ledger; time-filtered, timezone-aware statements and CSV export; idempotent month-tagged interest payments; a `Bank` registry that rejects duplicate account numbers; atomic transfers between its accounts using deterministic lock ordering; and extensive deterministic/concurrent tests.

**Scope:** this is an **in-memory educational model**, not a deployable banking platform. A production system needs persistent ACID ledgers, access control, durable unique identifiers, appropriate financial policies, and reliable idempotency across processes.

In [9]:
from __future__ import annotations

from dataclasses import dataclass, replace
from datetime import datetime, timedelta, timezone, tzinfo
from decimal import Decimal, InvalidOperation, ROUND_HALF_EVEN
from itertools import count
from io import StringIO
from threading import Lock, RLock
from typing import ClassVar
import csv
import re

CENT = Decimal("0.01")
MAX_BALANCE = Decimal("999999999999999.99")
CONFIRMATION_RE = re.compile(
    r"(?P<code>[DWIX])-(?P<account>[0-9]+)-(?P<stamp>[0-9]{14})-(?P<id>[1-9][0-9]*)"
)
PERIOD_RE = re.compile(r"(?!0000)[0-9]{4}-(?:0[1-9]|1[0-2])")


def as_decimal(value: Decimal | str | int | float, *, label: str) -> Decimal:
    """Turn supported inputs into a *finite* Decimal; never use Decimal(float)."""
    if isinstance(value, bool) or not isinstance(value, (Decimal, str, int, float)):
        raise TypeError(f"{label} must be Decimal, str, int, or float (not bool)")
    try:
        number = Decimal(str(value))
    except (InvalidOperation, ValueError) as exc:
        raise ValueError(f"{label} must be a valid number") from exc
    if not number.is_finite():
        raise ValueError(f"{label} must be finite")
    return number


def money(value: Decimal | str | int | float, *, label: str) -> Decimal:
    """Validate user-specified monetary amounts; never silently round inputs."""
    number = as_decimal(value, label=label)
    try:
        rounded = number.quantize(CENT, rounding=ROUND_HALF_EVEN)
    except InvalidOperation as exc:
        raise ValueError(f"{label} is outside the supported currency range") from exc
    if rounded != number:
        raise ValueError(f"{label} cannot have fractions of a cent")
    if abs(rounded) > MAX_BALANCE:
        raise ValueError(f"{label} exceeds the supported currency range")
    return rounded


def _aware_utc(value: datetime, *, label: str) -> datetime:
    if not isinstance(value, datetime) or value.tzinfo is None or value.utcoffset() is None:
        raise ValueError(f"{label} must be a timezone-aware datetime")
    return value.astimezone(timezone.utc)

In [10]:
@dataclass(frozen=True, slots=True)
class TimeZone:
    """Fixed-offset named timezone; does NOT perform daylight-saving transitions."""

    name: str
    offset_hours: int = 0
    offset_minutes: int = 0

    def __post_init__(self) -> None:
        if not isinstance(self.name, str) or not self.name.strip():
            raise ValueError("Timezone name must be nonempty")
        if type(self.offset_hours) is not int or type(self.offset_minutes) is not int:
            raise TypeError("Timezone offsets must be integers")
        if abs(self.offset_minutes) >= 60:
            raise ValueError("Minute offset must be strictly less than 60")
        if self.offset_hours and self.offset_minutes and (
            (self.offset_hours > 0) != (self.offset_minutes > 0)
        ):
            raise ValueError("Nonzero hour and minute offsets must have the same sign")
        total_minutes = self.offset_hours * 60 + self.offset_minutes
        if abs(total_minutes) > 14 * 60:
            raise ValueError("UTC offset must be between -14:00 and +14:00")
        object.__setattr__(self, "name", self.name.strip())

    @property
    def tzinfo(self) -> timezone:
        return timezone(timedelta(hours=self.offset_hours, minutes=self.offset_minutes), self.name)


UTC = TimeZone("UTC")
MST = TimeZone("MST", -7)


def resolve_timezone(zone: TimeZone | tzinfo) -> tzinfo:
    result = zone.tzinfo if isinstance(zone, TimeZone) else zone
    if not isinstance(result, tzinfo):
        raise TypeError("Timezone must be a TimeZone or datetime.tzinfo")
    return result


@dataclass(frozen=True, slots=True)
class Confirmation:
    account_number: str
    transaction_code: str
    transaction_id: int
    timestamp_utc: datetime
    timestamp_local: datetime

    @property
    def time_utc(self) -> str:
        """The original exercise's UTC presentation, without a trailing Z."""
        return self.timestamp_utc.strftime("%Y-%m-%dT%H:%M:%S")

    @property
    def time(self) -> str:
        label = self.timestamp_local.tzname() or self.timestamp_local.strftime("%z")
        return f"{self.timestamp_local:%Y-%m-%d %H:%M:%S} ({label})"


@dataclass(frozen=True, slots=True)
class Transaction:
    transaction_id: int
    code: str
    amount: Decimal
    balance_before: Decimal
    balance_after: Decimal
    timestamp_utc: datetime
    confirmation: str
    reason: str | None = None
    related_confirmation: str | None = None
    interest_period: str | None = None


@dataclass(frozen=True, slots=True)
class StatementEntry:
    transaction: Transaction
    local_time: datetime


@dataclass(frozen=True, slots=True)
class AccountStatement:
    account_number: str
    opening_balance: Decimal
    closing_balance: Decimal
    deposits: Decimal
    withdrawals: Decimal
    interest: Decimal
    declined_count: int
    entries: tuple[StatementEntry, ...]


@dataclass(frozen=True, slots=True)
class TransferReceipt:
    accepted: bool
    source_account_number: str
    destination_account_number: str
    amount: Decimal
    withdrawal_confirmation: str
    deposit_confirmation: str | None

## 1. Account: encapsulation, monetary rules, UTC receipts, and an immutable ledger

The ledger is an immutable **snapshot** when accessed through `transactions`; transaction objects are frozen dataclasses. The balance and the ledger are changed while holding the **same per-account lock**. Private attributes are a Python convention, not an authorization boundary.

The global counter is lock-protected and referenced via `Account` rather than `cls`, so subclasses do not accidentally fork the ID sequence. IDs restart on a new process, as allowed by the exercise. The class-wide interest rate is accessed through class methods to make sharing explicit.

In [11]:
class Account:
    """Thread-safe, in-memory account with conventional (not security) encapsulation."""

    _monthly_interest_rate: ClassVar[Decimal] = Decimal("0.005")
    _rate_lock: ClassVar[Lock] = Lock()
    _id_lock: ClassVar[Lock] = Lock()
    _transaction_ids: ClassVar[count] = count(1)

    def __init__(
        self,
        account_number: str | int,
        first_name: str,
        last_name: str,
        preferred_timezone: TimeZone = UTC,
        starting_balance: Decimal | str | int | float = "0.00",
    ) -> None:
        identifier = self._validate_identifier(account_number)
        if not isinstance(preferred_timezone, TimeZone):
            raise TypeError("preferred_timezone must be a TimeZone")
        opening = money(starting_balance, label="Starting balance")
        if opening < 0:
            raise ValueError("Starting balance cannot be negative")
        self._lock = RLock()
        self._account_number = identifier
        self._first_name = self._validate_name(first_name, "First name")
        self._last_name = self._validate_name(last_name, "Last name")
        self._preferred_timezone = preferred_timezone
        self._opening_balance = opening
        self._balance = opening
        self._transactions: list[Transaction] = []
        self._paid_periods: dict[str, str] = {}
        self._bank: Bank | None = None  # Set by Bank.add_account under account lock.

    @staticmethod
    def _validate_identifier(value: str | int) -> str:
        if isinstance(value, bool) or not isinstance(value, (str, int)):
            raise TypeError("Account number must be a digit-only string or integer")
        identifier = str(value)
        if not identifier or not identifier.isascii() or not identifier.isdecimal():
            raise ValueError("Account number must contain only ASCII digits")
        return identifier

    @staticmethod
    def _validate_name(value: str, label: str) -> str:
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"{label} must be a nonempty string")
        return value.strip()

    @staticmethod
    def _utcnow() -> datetime:
        return datetime.now(timezone.utc)

    @classmethod
    def _next_id(cls) -> int:
        with Account._id_lock:
            return next(Account._transaction_ids)

    @classmethod
    def get_monthly_interest_rate(cls) -> Decimal:
        with Account._rate_lock:
            return Account._monthly_interest_rate

    @classmethod
    def set_monthly_interest_rate(cls, rate: Decimal | str | int | float) -> None:
        """Fractional rate: 0.005 is 0.5%, not 5%. Applies to ALL accounts."""
        value = as_decimal(rate, label="Monthly interest rate")
        if value < 0:
            raise ValueError("Monthly interest rate cannot be negative")
        with Account._rate_lock:
            Account._monthly_interest_rate = value

    @property
    def interest_rate(self) -> Decimal:
        return self.get_monthly_interest_rate()

    @property
    def account_number(self) -> str:
        return self._account_number

    @property
    def first_name(self) -> str:
        with self._lock:
            return self._first_name

    @first_name.setter
    def first_name(self, value: str) -> None:
        validated = self._validate_name(value, "First name")
        with self._lock:
            self._first_name = validated

    @property
    def last_name(self) -> str:
        with self._lock:
            return self._last_name

    @last_name.setter
    def last_name(self, value: str) -> None:
        validated = self._validate_name(value, "Last name")
        with self._lock:
            self._last_name = validated

    @property
    def full_name(self) -> str:
        with self._lock:
            return f"{self._first_name} {self._last_name}"

    @property
    def preferred_timezone(self) -> TimeZone:
        with self._lock:
            return self._preferred_timezone

    @preferred_timezone.setter
    def preferred_timezone(self, value: TimeZone) -> None:
        if not isinstance(value, TimeZone):
            raise TypeError("preferred_timezone must be a TimeZone")
        with self._lock:
            self._preferred_timezone = value

    @property
    def balance(self) -> Decimal:
        with self._lock:
            return self._balance

    @property
    def transactions(self) -> tuple[Transaction, ...]:
        """Immutable, coherent snapshot; no writable internal list is returned."""
        with self._lock:
            return tuple(self._transactions)

    def _record(
        self,
        code: str,
        amount: Decimal,
        before: Decimal,
        *,
        reason: str | None = None,
        related_confirmation: str | None = None,
        interest_period: str | None = None,
    ) -> str:
        """Private: caller MUST hold self._lock; called after balance mutation."""
        timestamp = _aware_utc(self._utcnow(), label="Transaction timestamp").replace(
            microsecond=0
        )
        transaction_id = self._next_id()
        confirmation = f"{code}-{self.account_number}-{timestamp:%Y%m%d%H%M%S}-{transaction_id}"
        self._transactions.append(
            Transaction(
                transaction_id, code, amount, before, self._balance, timestamp,
                confirmation, reason, related_confirmation, interest_period,
            )
        )
        return confirmation

    def _post(
        self, code: str, amount: Decimal, before: Decimal, after: Decimal,
        *, interest_period: str | None = None,
    ) -> str:
        """Private: commit balance plus ledger together, reverting if recording fails."""
        original_length = len(self._transactions)
        self._balance = after
        try:
            return self._record(code, amount, before, interest_period=interest_period)
        except Exception:
            self._balance = before
            del self._transactions[original_length:]
            raise

    def deposit(self, amount: Decimal | str | int | float) -> str:
        value = money(amount, label="Deposit")
        if value <= 0:
            raise ValueError("Deposit must be strictly positive")
        with self._lock:
            before = self._balance
            after = before + value
            if after > MAX_BALANCE:
                raise ValueError("Deposit would exceed maximum supported balance")
            return self._post("D", value, before, after)

    def withdraw(self, amount: Decimal | str | int | float) -> str:
        value = money(amount, label="Withdrawal")
        if value <= 0:
            raise ValueError("Withdrawal must be strictly positive")
        with self._lock:
            before = self._balance
            if value > before:
                return self._record("X", value, before, reason="Insufficient funds")
            return self._post("W", value, before, before - value)

    def pay_interest(self, period: str | None = None) -> str:
        """Credit monthly interest; optional YYYY-MM makes the call idempotent per month.

        Without a period, every call credits interest (backward-compatible exercise behavior).
        With a period, repeat calls return the ORIGINAL confirmation, not a second credit.
        A zero-interest payment is still recorded as an I transaction.
        """
        if period is not None and (
            not isinstance(period, str) or PERIOD_RE.fullmatch(period) is None
        ):
            raise ValueError("period must be YYYY-MM (e.g. '2026-09')")
        with self._lock:
            if period is not None and period in self._paid_periods:
                return self._paid_periods[period]
            rate = self.get_monthly_interest_rate()
            before = self._balance
            try:
                interest = (before * rate).quantize(CENT, rounding=ROUND_HALF_EVEN)
            except InvalidOperation as exc:
                raise ValueError("Calculated interest is outside the supported range") from exc
            after = before + interest
            if after > MAX_BALANCE:
                raise ValueError("Interest would exceed maximum supported balance")
            confirmation = self._post("I", interest, before, after, interest_period=period)
            if period is not None:
                self._paid_periods[period] = confirmation
            return confirmation

    @staticmethod
    def parse_confirmation(
        confirmation: str, desired_timezone: TimeZone | tzinfo = UTC
    ) -> Confirmation:
        """Parse a receipt *structurally*; does not establish that it is authentic."""
        if not isinstance(confirmation, str):
            raise TypeError("Confirmation must be a string")
        match = CONFIRMATION_RE.fullmatch(confirmation)
        if match is None:
            raise ValueError("Malformed confirmation number")
        zone = resolve_timezone(desired_timezone)
        try:
            stamp = datetime.strptime(match["stamp"], "%Y%m%d%H%M%S").replace(
                tzinfo=timezone.utc
            )
        except ValueError as exc:
            raise ValueError("Confirmation contains an invalid UTC timestamp") from exc
        return Confirmation(
            account_number=match["account"],
            transaction_code=match["code"],
            transaction_id=int(match["id"]),
            timestamp_utc=stamp,
            timestamp_local=stamp.astimezone(zone),
        )

    def get_transaction(self, confirmation: str) -> Transaction:
        """Search this account's own ledger; malformed or unknown codes raise KeyError."""
        with self._lock:
            for entry in self._transactions:
                if entry.confirmation == confirmation:
                    return entry
        raise KeyError("Confirmation not present in this account's ledger")

    def verify_confirmation(self, confirmation: str) -> bool:
        """Confirm a receipt exists in this *in-memory* ledger (not digital signing)."""
        try:
            self.get_transaction(confirmation)
        except KeyError:
            return False
        return True

    def statement(
        self,
        start: datetime | None = None,
        end: datetime | None = None,
        desired_timezone: TimeZone | tzinfo | None = None,
    ) -> AccountStatement:
        """Period [start, end), based on aware instants; no dates = all transactions."""
        beginning = None if start is None else _aware_utc(start, label="start")
        ending = None if end is None else _aware_utc(end, label="end")
        if beginning is not None and ending is not None and beginning >= ending:
            raise ValueError("start must precede end")
        zone = resolve_timezone(
            self.preferred_timezone if desired_timezone is None else desired_timezone
        )
        with self._lock:
            snapshot = tuple(self._transactions)
            opening = self._opening_balance
        entries: list[StatementEntry] = []
        deposits = withdrawals = interest = Decimal("0.00")
        declined = 0
        closing = opening
        for tx in snapshot:
            if beginning is not None and tx.timestamp_utc < beginning:
                opening = tx.balance_after
                closing = tx.balance_after
                continue
            if ending is not None and tx.timestamp_utc >= ending:
                continue
            entries.append(StatementEntry(tx, tx.timestamp_utc.astimezone(zone)))
            closing = tx.balance_after
            if tx.code == "D":
                deposits += tx.amount
            elif tx.code == "W":
                withdrawals += tx.amount
            elif tx.code == "I":
                interest += tx.amount
            elif tx.code == "X":
                declined += 1
        return AccountStatement(
            self.account_number, opening, closing, deposits, withdrawals,
            interest, declined, tuple(entries),
        )

    def export_statement_csv(
        self,
        start: datetime | None = None,
        end: datetime | None = None,
        desired_timezone: TimeZone | tzinfo | None = None,
    ) -> str:
        """Return CSV data; no filesystem side effects. Times retain UTC offsets."""
        statement = self.statement(start, end, desired_timezone)
        output = StringIO(newline="")
        writer = csv.writer(output)
        writer.writerow(
            ("transaction_id", "timestamp_local", "timestamp_utc", "code", "amount",
             "balance_before", "balance_after", "confirmation", "reason",
             "related_confirmation", "interest_period")
        )
        for entry in statement.entries:
            tx = entry.transaction
            writer.writerow(
                (tx.transaction_id, entry.local_time.isoformat(), tx.timestamp_utc.isoformat(),
                 tx.code, str(tx.amount), str(tx.balance_before), str(tx.balance_after),
                 tx.confirmation, tx.reason or "", tx.related_confirmation or "",
                 tx.interest_period or "")
            )
        return output.getvalue()

    def audit(self) -> bool:
        """Check basic local ledger arithmetic and confirmation integrity."""
        with self._lock:
            running = self._opening_balance
            seen: set[int] = set()
            for tx in self._transactions:
                if tx.transaction_id in seen or tx.balance_before != running:
                    return False
                seen.add(tx.transaction_id)
                if tx.code == "D" or tx.code == "I":
                    expected = running + tx.amount
                elif tx.code == "W":
                    expected = running - tx.amount
                elif tx.code == "X":
                    expected = running
                else:
                    return False
                if expected != tx.balance_after or expected < 0 or expected > MAX_BALANCE:
                    return False
                try:
                    parsed = self.parse_confirmation(tx.confirmation)
                except (TypeError, ValueError, OverflowError):
                    return False
                if (
                    parsed.account_number != self.account_number
                    or parsed.transaction_id != tx.transaction_id
                    or parsed.transaction_code != tx.code
                    or parsed.timestamp_utc != tx.timestamp_utc
                ):
                    return False
                running = expected
            return running == self._balance

## 2. Bank registry and atomic transfers (optional extension)

`Bank` owns a strong-reference registry and rejects duplicate account numbers. Creating independent `Account(...)` objects directly is still supported for the original exercise; **uniqueness is enforced within each `Bank` registry**, not across disconnected banks or program restarts.

A transfer acquires both account locks **in sorted account-number order**, preventing circular waits when two concurrent transfers run in opposite directions. An accepted transfer records a linked `W` and `D`; insufficient funds record only `X` and do not credit the recipient. If an unexpected in-memory error occurs between debit and credit, balances and ledger entries are rolled back and the error propagates. The global ID sequence is intentionally **not** rolled back; sequence gaps after failures are acceptable. This does **not** replace a database transaction.

In [12]:
class Bank:
    """In-memory account registry; one account number per bank."""

    def __init__(self) -> None:
        self._lock = RLock()
        self._accounts: dict[str, Account] = {}

    def add_account(self, account: Account) -> Account:
        if not isinstance(account, Account):
            raise TypeError("account must be an Account")
        with self._lock, account._lock:
            if account.account_number in self._accounts:
                raise ValueError(f"Duplicate account number: {account.account_number}")
            if account._bank is not None:
                raise ValueError("Account already belongs to a bank")
            self._accounts[account.account_number] = account
            account._bank = self
        return account

    def open_account(
        self,
        account_number: str | int,
        first_name: str,
        last_name: str,
        preferred_timezone: TimeZone = UTC,
        starting_balance: Decimal | str | int | float = "0.00",
    ) -> Account:
        """Atomically register a new account; duplicates fail with ValueError."""
        identifier = Account._validate_identifier(account_number)
        with self._lock:
            if identifier in self._accounts:
                raise ValueError(f"Duplicate account number: {identifier}")
            account = Account(
                identifier, first_name, last_name, preferred_timezone, starting_balance
            )
            return self.add_account(account)

    def get_account(self, account_number: str | int) -> Account:
        identifier = Account._validate_identifier(account_number)
        with self._lock:
            try:
                return self._accounts[identifier]
            except KeyError as exc:
                raise KeyError(f"Unknown account: {identifier}") from exc

    @property
    def accounts(self) -> tuple[Account, ...]:
        with self._lock:
            return tuple(self._accounts[key] for key in sorted(self._accounts))

    def transfer(
        self,
        source_account_number: str | int,
        destination_account_number: str | int,
        amount: Decimal | str | int | float,
    ) -> TransferReceipt:
        value = money(amount, label="Transfer")
        if value <= 0:
            raise ValueError("Transfer must be strictly positive")
        with self._lock:
            source = self.get_account(source_account_number)
            destination = self.get_account(destination_account_number)
            if source is destination:
                raise ValueError("Source and destination must differ")
        first, second = sorted((source, destination), key=lambda a: a.account_number)
        with first._lock, second._lock:
            if source._balance < value:
                declined = source._record(
                    "X", value, source._balance, reason="Insufficient funds for transfer"
                )
                return TransferReceipt(
                    False, source.account_number, destination.account_number,
                    value, declined, None,
                )
            if destination._balance + value > MAX_BALANCE:
                declined = source._record(
                    "X", value, source._balance, reason="Destination balance limit"
                )
                return TransferReceipt(
                    False, source.account_number, destination.account_number,
                    value, declined, None,
                )
            source_before, destination_before = source._balance, destination._balance
            source_length, destination_length = (
                len(source._transactions), len(destination._transactions)
            )
            try:
                source._balance -= value
                debit = source._record("W", value, source_before)
                destination._balance += value
                credit = destination._record(
                    "D", value, destination_before, related_confirmation=debit
                )
                source._transactions[-1] = replace(
                    source._transactions[-1], related_confirmation=credit
                )
            except Exception:
                source._balance, destination._balance = source_before, destination_before
                del source._transactions[source_length:]
                del destination._transactions[destination_length:]
                raise
            return TransferReceipt(
                True, source.account_number, destination.account_number, value, debit, credit
            )

    def audit(self) -> bool:
        """Check account audit chains and uniqueness of IDs across this bank's ledgers."""
        snapshot = self.accounts
        if not all(account.audit() for account in snapshot):
            return False
        ids = [tx.transaction_id for account in snapshot for tx in account.transactions]
        return len(ids) == len(set(ids))

## 3. Worked examples and confirmation parsing

A confirmation includes the account number, UTC time, type, and global ID; timezone conversion must be decided by the **caller**. `ZoneInfo` can be used instead of fixed offsets when real daylight-saving rules are needed, assuming IANA timezone data is installed.

In [13]:
bank = Bank()
account = bank.open_account("140568", "Ada", "Lovelace", MST, "100.00")
destination = bank.open_account("000140569", "Grace", "Hopper", UTC, "30.00")

confirmation = account.deposit("50.00")
parsed = Account.parse_confirmation(confirmation, MST)
print("Account:", account.account_number, account.full_name)
print("Deposit confirmation:", confirmation)
print("Parsed code:", parsed.transaction_code)
print("UTC:", parsed.time_utc, "| Preferred local:", parsed.time)
print("Balance:", account.balance)

rejected = account.withdraw("999.00")
print("Declined confirmation:", rejected)
print("Balance after decline:", account.balance)
print("Valid ledger receipt:", account.verify_confirmation(confirmation))
print("Fabricated receipt valid?:", account.verify_confirmation("D-140568-20190315145900-999999"))

transfer = bank.transfer("140568", "000140569", "25.00")
print("Transfer:", transfer)
print("Balances after transfer:", account.balance, destination.balance)
print("Both account audit checks passed:", bank.audit())
assert account.balance == Decimal("125.00")
assert destination.balance == Decimal("55.00")
assert bank.audit()

Account: 140568 Ada Lovelace
Deposit confirmation: D-140568-20260920123625-1
Parsed code: D
UTC: 2026-09-20T12:36:25 | Preferred local: 2026-09-20 05:36:25 (MST)
Balance: 150.00
Declined confirmation: X-140568-20260920123625-2
Balance after decline: 150.00
Valid ledger receipt: True
Fabricated receipt valid?: False
Transfer: TransferReceipt(accepted=True, source_account_number='140568', destination_account_number='000140569', amount=Decimal('25.00'), withdrawal_confirmation='W-140568-20260920123625-3', deposit_confirmation='D-000140569-20260920123625-4')
Balances after transfer: 125.00 55.00
Both account audit checks passed: True


In [14]:
# Correct interest arithmetic. Restore the global interest rate so later cells are isolated.
old_rate = Account.get_monthly_interest_rate()
try:
    Account.set_monthly_interest_rate("0.005")  # 0.5%
    small_rate_account = bank.open_account("000140570", "Katherine", "Johnson", starting_balance="1000")
    first_interest = small_rate_account.pay_interest("2026-09")
    repeated_interest = small_rate_account.pay_interest("2026-09")
    print("0.5% produces:", small_rate_account.balance, "| Same receipt on retry:", first_interest == repeated_interest)
    assert small_rate_account.balance == Decimal("1005.00")
    assert first_interest == repeated_interest

    Account.set_monthly_interest_rate("0.05")  # 5%
    larger_rate_account = bank.open_account("000140571", "Alan", "Turing", starting_balance="1000")
    print("5% produces:", larger_rate_account.pay_interest("2026-09"), larger_rate_account.balance)
    assert larger_rate_account.balance == Decimal("1050.00")
finally:
    Account.set_monthly_interest_rate(old_rate)

statement = account.statement(desired_timezone=MST)
print("Statement:", statement)
print("Statement CSV (first 3 lines):")
print("\n".join(account.export_statement_csv().splitlines()[:3]))

0.5% produces: 1005.00 | Same receipt on retry: True
5% produces: I-000140571-20260920123625-6 1050.00
Statement: AccountStatement(account_number='140568', opening_balance=Decimal('100.00'), closing_balance=Decimal('125.00'), deposits=Decimal('50.00'), withdrawals=Decimal('25.00'), interest=Decimal('0.00'), declined_count=1, entries=(StatementEntry(transaction=Transaction(transaction_id=1, code='D', amount=Decimal('50.00'), balance_before=Decimal('100.00'), balance_after=Decimal('150.00'), timestamp_utc=datetime.datetime(2026, 9, 20, 12, 36, 25, tzinfo=datetime.timezone.utc), confirmation='D-140568-20260920123625-1', reason=None, related_confirmation=None, interest_period=None), local_time=datetime.datetime(2026, 9, 20, 5, 36, 25, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200), 'MST'))), StatementEntry(transaction=Transaction(transaction_id=2, code='X', amount=Decimal('999.00'), balance_before=Decimal('150.00'), balance_after=Decimal('150.00'), timestamp_utc=dateti

In [15]:
# Optional real-location timezone conversion (daylight saving is handled by ZoneInfo).
try:
    from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
    ny = ZoneInfo("America/New_York")
except ZoneInfoNotFoundError:
    print("IANA timezone database unavailable; fixed-offset TimeZone still works.")
else:
    known_receipt = "D-140568-20190315145900-124"
    print("New York on 2019-03-15:", Account.parse_confirmation(known_receipt, ny).time)
    # March 15, 2019 uses EDT (UTC-04:00), rather than a permanent fixed offset.
    assert Account.parse_confirmation(known_receipt, ny).time.startswith("2019-03-15 10:59:00")

New York on 2019-03-15: 2019-03-15 10:59:00 (EDT)


## 4. Automated tests: validation, edge cases, concurrency, audit, reporting

Tests do not assume fixed global transaction IDs because they are shared across every account in the live kernel. A deterministic patched UTC clock checks exact receipt structure; cross-direction concurrent transfers test lock ordering and conservation of money. Interest-rate mutations are restored after each test. Run the next cell to execute all tests.

In [16]:
import unittest
from concurrent.futures import ThreadPoolExecutor
from unittest.mock import patch


class BankAccountTests(unittest.TestCase):
    def setUp(self) -> None:
        self.old_rate = Account.get_monthly_interest_rate()
        Account.set_monthly_interest_rate("0.005")
        self.bank = Bank()
        self.a = self.bank.open_account("00140568", " Ada ", "Lovelace", MST, "100.00")
        self.b = self.bank.open_account("00140569", "Grace", "Hopper", UTC, "50.00")

    def tearDown(self) -> None:
        Account.set_monthly_interest_rate(self.old_rate)

    def test_01_properties_and_names(self) -> None:
        self.assertEqual(self.a.account_number, "00140568")
        self.assertEqual(self.a.full_name, "Ada Lovelace")
        self.a.first_name = " Katherine "
        self.assertEqual(self.a.full_name, "Katherine Lovelace")
        with self.assertRaises(AttributeError):
            self.a.balance = Decimal("999.00")
        with self.assertRaises(AttributeError):
            self.a.account_number = "2"
        with self.assertRaises(ValueError):
            self.a.last_name = "   "
        self.a.preferred_timezone = UTC
        self.assertEqual(self.a.preferred_timezone, UTC)

    def test_02_account_validation(self) -> None:
        for bad in ("", "-123", "1a", "１２３", True, 12.5):
            with self.subTest(value=bad), self.assertRaises((ValueError, TypeError)):
                Account(bad, "A", "B")
        with self.assertRaises(ValueError):
            Account("13", "A", "B", starting_balance="-0.01")
        with self.assertRaises(TypeError):
            Account("13", "A", "B", preferred_timezone=timezone.utc)
        with self.assertRaises(ValueError):
            Account("13", " ", "B")

    def test_03_money_inputs(self) -> None:
        for bad in ("0", "-1", "0.001", "NaN", "Infinity", "1000000000000000"):
            with self.subTest(value=bad), self.assertRaises(ValueError):
                self.a.deposit(bad)
        for bad in (True, None, object()):
            with self.subTest(value=bad), self.assertRaises(TypeError):
                self.a.withdraw(bad)
        self.assertEqual(self.a.balance, Decimal("100.00"))
        self.assertEqual(len(self.a.transactions), 0)

    def test_04_deposit_withdraw_and_before_after_balances(self) -> None:
        d = self.a.deposit("50")
        w = self.a.withdraw("25")
        self.assertTrue(d.startswith("D-00140568-"))
        self.assertTrue(w.startswith("W-00140568-"))
        self.assertEqual(self.a.balance, Decimal("125.00"))
        self.assertEqual(self.a.transactions[0].balance_before, Decimal("100.00"))
        self.assertEqual(self.a.transactions[-1].balance_after, Decimal("125.00"))
        self.assertTrue(self.a.audit())

    def test_05_declined_withdrawal(self) -> None:
        receipt = self.a.withdraw("100.01")
        self.assertTrue(receipt.startswith("X-"))
        self.assertEqual(self.a.balance, Decimal("100.00"))
        self.assertEqual(self.a.get_transaction(receipt).reason, "Insufficient funds")
        self.assertEqual(self.a.get_transaction(receipt).balance_before, self.a.balance)

    def test_06_shared_interest_and_prompt_correction(self) -> None:
        Account.set_monthly_interest_rate("0.005")
        p = Account("99", "A", "B", starting_balance="1000")
        p.pay_interest()
        self.assertEqual(p.balance, Decimal("1005.00"))
        Account.set_monthly_interest_rate("0.05")
        q = Account("98", "A", "B", starting_balance="1000")
        q.pay_interest()
        self.assertEqual(q.balance, Decimal("1050.00"))
        self.assertEqual(self.a.interest_rate, q.interest_rate)
        with self.assertRaises(ValueError):
            Account.set_monthly_interest_rate("-0.01")

    def test_07_half_even_rounding_and_zero_interest(self) -> None:
        Account.set_monthly_interest_rate("0.025")
        tiny = Account("1", "A", "B", starting_balance="0.20")
        confirmation = tiny.pay_interest()
        self.assertEqual(tiny.balance, Decimal("0.20"))  # 0.005 -> 0.00
        self.assertEqual(tiny.get_transaction(confirmation).amount, Decimal("0.00"))
        empty = Account("2", "A", "B")
        self.assertTrue(empty.pay_interest().startswith("I-"))

    def test_08_interest_idempotency(self) -> None:
        first = self.a.pay_interest("2026-09")
        second = self.a.pay_interest("2026-09")
        self.assertEqual(first, second)
        self.assertEqual(len(self.a.transactions), 1)
        self.assertEqual(self.a.balance, Decimal("100.50"))
        third = self.a.pay_interest("2026-10")
        self.assertNotEqual(first, third)
        self.assertEqual(self.a.balance, Decimal("101.00"))  # 100.50 * 0.005 = 0.5025 -> 0.50
        for bad in ("2026-13", "2026-9", 202609, "bad"):
            with self.subTest(period=bad), self.assertRaises(ValueError):
                self.a.pay_interest(bad)

    def test_09_interest_idempotency_under_concurrency(self) -> None:
        with ThreadPoolExecutor(max_workers=8) as pool:
            receipts = list(pool.map(lambda _: self.a.pay_interest("2026-09"), range(100)))
        self.assertEqual(len(set(receipts)), 1)
        self.assertEqual(len(self.a.transactions), 1)
        self.assertEqual(self.a.balance, Decimal("100.50"))

    def test_10_parse_exact_known_timestamp(self) -> None:
        instant = datetime(2019, 3, 15, 14, 59, tzinfo=timezone.utc)
        with patch.object(Account, "_utcnow", return_value=instant):
            receipt = self.a.deposit("50.00")
        result = Account.parse_confirmation(receipt, MST)
        self.assertEqual(result.account_number, "00140568")
        self.assertEqual(result.transaction_code, "D")
        self.assertEqual(result.time_utc, "2019-03-15T14:59:00")
        self.assertEqual(result.time, "2019-03-15 07:59:00 (MST)")
        self.assertTrue(receipt.startswith("D-00140568-20190315145900-"))
        self.assertEqual(result.timestamp_utc.tzinfo, timezone.utc)
        self.assertEqual(result.transaction_id, self.a.transactions[-1].transaction_id)

    def test_11_parser_rejects_malformed_receipts(self) -> None:
        for receipt in ("bad", "D-1-20190230010101-1", "Z-1-20200101000000-1", "D-1-20200101000000-0"):
            with self.subTest(receipt=receipt), self.assertRaises(ValueError):
                Account.parse_confirmation(receipt)
        with self.assertRaises(TypeError):
            Account.parse_confirmation("D-1-20200101000000-1", "MST")
        with self.assertRaises(TypeError):
            Account.parse_confirmation(None)

    def test_12_fixed_timezone_validation(self) -> None:
        with self.assertRaises(ValueError):
            TimeZone("bad", 15)
        with self.assertRaises(ValueError):
            TimeZone("bad", -3, 30)
        with self.assertRaises(ValueError):
            TimeZone("bad", 0, 60)
        self.assertEqual(TimeZone("NPT", 5, 45).tzinfo.utcoffset(None), timedelta(hours=5, minutes=45))
        self.assertEqual(TimeZone("Newfoundland", -3, -30).tzinfo.utcoffset(None), timedelta(hours=-3, minutes=-30))

    def test_13_bank_prevents_duplicate_account_ids(self) -> None:
        with self.assertRaises(ValueError):
            self.bank.open_account("00140568", "Other", "Holder")
        with self.assertRaises(ValueError):
            self.bank.add_account(Account("00140568", "Other", "Holder"))
        different_bank = Bank()
        with self.assertRaises(ValueError):
            different_bank.add_account(self.a)
        self.assertIs(self.bank.get_account("00140568"), self.a)
        with self.assertRaises(KeyError):
            self.bank.get_account("00149999")
        self.assertEqual(len(self.bank.accounts), 2)

    def test_14_successful_transfer_and_linked_receipts(self) -> None:
        receipt = self.bank.transfer("00140568", "00140569", "30")
        self.assertTrue(receipt.accepted)
        self.assertEqual(self.a.balance, Decimal("70.00"))
        self.assertEqual(self.b.balance, Decimal("80.00"))
        self.assertTrue(receipt.withdrawal_confirmation.startswith("W-"))
        self.assertTrue(receipt.deposit_confirmation.startswith("D-"))
        self.assertEqual(
            self.a.get_transaction(receipt.withdrawal_confirmation).related_confirmation,
            receipt.deposit_confirmation,
        )
        self.assertEqual(
            self.b.get_transaction(receipt.deposit_confirmation).related_confirmation,
            receipt.withdrawal_confirmation,
        )
        self.assertTrue(self.bank.audit())

    def test_15_declined_transfer_does_not_credit(self) -> None:
        receipt = self.bank.transfer("00140568", "00140569", "999")
        self.assertFalse(receipt.accepted)
        self.assertIsNone(receipt.deposit_confirmation)
        self.assertEqual(self.a.balance, Decimal("100.00"))
        self.assertEqual(self.b.balance, Decimal("50.00"))
        self.assertEqual(self.a.get_transaction(receipt.withdrawal_confirmation).code, "X")
        self.assertEqual(len(self.b.transactions), 0)

    def test_16_transfer_rejects_invalid_targets(self) -> None:
        with self.assertRaises(ValueError):
            self.bank.transfer("00140568", "00140568", "1")
        with self.assertRaises(KeyError):
            self.bank.transfer("00140568", "00149999", "1")
        with self.assertRaises(ValueError):
            self.bank.transfer("00140568", "00140569", "0")

    def test_17_statement_totals_and_boundaries(self) -> None:
        day1 = datetime(2026, 9, 1, 0, tzinfo=timezone.utc)
        day2 = datetime(2026, 9, 2, 0, tzinfo=timezone.utc)
        day3 = datetime(2026, 9, 3, 0, tzinfo=timezone.utc)
        with patch.object(Account, "_utcnow", return_value=day1):
            self.a.deposit("10")
        with patch.object(Account, "_utcnow", return_value=day2):
            self.a.withdraw("3")
            self.a.withdraw("999")
        with patch.object(Account, "_utcnow", return_value=day3):
            self.a.deposit("7")
        statement = self.a.statement(day2, day3)
        self.assertEqual(statement.opening_balance, Decimal("110.00"))
        self.assertEqual(statement.closing_balance, Decimal("107.00"))
        self.assertEqual(statement.withdrawals, Decimal("3.00"))
        self.assertEqual(statement.deposits, Decimal("0.00"))
        self.assertEqual(statement.declined_count, 1)
        self.assertEqual(len(statement.entries), 2)
        self.assertEqual(self.a.statement().closing_balance, Decimal("114.00"))
        with self.assertRaises(ValueError):
            self.a.statement(day3, day2)
        with self.assertRaises(ValueError):
            self.a.statement(datetime(2026, 9, 1))  # Naive dates forbidden

    def test_18_csv_export_and_immutable_ledger(self) -> None:
        code = self.a.deposit("1")
        csv_lines = list(csv.DictReader(StringIO(self.a.export_statement_csv())))
        self.assertEqual(len(csv_lines), 1)
        self.assertEqual(csv_lines[0]["confirmation"], code)
        self.assertEqual(csv_lines[0]["balance_after"], "101.00")
        self.assertIn("-07:00", csv_lines[0]["timestamp_local"])
        snapshot = self.a.transactions
        self.assertIsInstance(snapshot, tuple)
        with self.assertRaises(AttributeError):
            snapshot[0].amount = Decimal("999.00")
        self.assertTrue(self.a.verify_confirmation(code))
        self.assertFalse(self.b.verify_confirmation(code))
        with self.assertRaises(KeyError):
            self.b.get_transaction(code)

    def test_19_amount_and_balance_limit(self) -> None:
        full = Account("999", "A", "B", starting_balance=str(MAX_BALANCE))
        with self.assertRaises(ValueError):
            full.deposit("0.01")
        with self.assertRaises(ValueError):
            full.pay_interest()
        self.assertEqual(full.balance, MAX_BALANCE)
        self.assertEqual(len(full.transactions), 0)
        self.assertTrue(full.audit())

    def test_20_global_ids_and_concurrent_deposits(self) -> None:
        with ThreadPoolExecutor(max_workers=8) as pool:
            codes = list(pool.map(lambda i: (self.a if i % 2 else self.b).deposit("1"), range(200)))
        ids = [Account.parse_confirmation(code).transaction_id for code in codes]
        self.assertEqual(len(set(ids)), 200)
        self.assertEqual(self.a.balance, Decimal("200.00"))
        self.assertEqual(self.b.balance, Decimal("150.00"))
        self.assertTrue(self.bank.audit())

    def test_21_concurrent_opposite_transfers(self) -> None:
        def transfer(i: int) -> TransferReceipt:
            if i % 2:
                return self.bank.transfer(self.a.account_number, self.b.account_number, "1")
            return self.bank.transfer(self.b.account_number, self.a.account_number, "1")

        with ThreadPoolExecutor(max_workers=8) as pool:
            receipts = list(pool.map(transfer, range(200)))
        self.assertTrue(all(receipt.accepted for receipt in receipts))
        self.assertEqual(self.a.balance + self.b.balance, Decimal("150.00"))
        self.assertEqual(self.a.balance, Decimal("100.00"))
        self.assertEqual(self.b.balance, Decimal("50.00"))
        self.assertEqual(len(self.a.transactions) + len(self.b.transactions), 400)
        self.assertTrue(self.bank.audit())

    def test_22_roll_back_transfer_on_internal_failure(self) -> None:
        original = self.b._record
        with patch.object(self.b, "_record", side_effect=RuntimeError("Injected failure")):
            with self.assertRaisesRegex(RuntimeError, "Injected failure"):
                self.bank.transfer(self.a.account_number, self.b.account_number, "10")
        self.assertEqual(self.a.balance, Decimal("100.00"))
        self.assertEqual(self.b.balance, Decimal("50.00"))
        self.assertEqual(self.a.transactions, ())
        self.assertEqual(self.b.transactions, ())
        self.assertTrue(self.bank.audit())

    def test_23_recording_failure_reverts_single_account_updates(self) -> None:
        with patch.object(self.a, "_record", side_effect=RuntimeError("Injected failure")):
            with self.assertRaisesRegex(RuntimeError, "Injected failure"):
                self.a.deposit("10")
            with self.assertRaisesRegex(RuntimeError, "Injected failure"):
                self.a.withdraw("10")
            with self.assertRaisesRegex(RuntimeError, "Injected failure"):
                self.a.pay_interest("2026-09")
        self.assertEqual(self.a.balance, Decimal("100.00"))
        self.assertEqual(self.a.transactions, ())
        self.assertTrue(self.a.audit())
        # An unsuccessful posting must not mark an interest period as paid.
        self.assertTrue(self.a.pay_interest("2026-09").startswith("I-"))

    def test_24_concurrent_registry_open_does_not_allow_duplicates(self) -> None:
        def attempt(_: int) -> bool:
            try:
                self.bank.open_account("900000", "A", "B")
                return True
            except ValueError:
                return False

        with ThreadPoolExecutor(max_workers=8) as pool:
            successful = list(pool.map(attempt, range(40)))
        self.assertEqual(sum(successful), 1)
        self.assertEqual(len(self.bank.accounts), 3)

    def test_25_audit_detects_accidental_ledger_corruption(self) -> None:
        self.a.deposit("1")
        self.assertTrue(self.a.audit())
        original = self.a._transactions[-1]
        try:
            self.a._transactions[-1] = replace(original, balance_after=Decimal("777.00"))
            self.assertFalse(self.a.audit())
        finally:
            self.a._transactions[-1] = original
        self.assertTrue(self.a.audit())


suite = unittest.defaultTestLoader.loadTestsFromTestCase(BankAccountTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "At least one automated test failed"
print(f"\nSUCCESS: {result.testsRun} automated tests passed.")

test_01_properties_and_names (__main__.BankAccountTests.test_01_properties_and_names) ... ok
test_02_account_validation (__main__.BankAccountTests.test_02_account_validation) ... ok
test_03_money_inputs (__main__.BankAccountTests.test_03_money_inputs) ... ok
test_04_deposit_withdraw_and_before_after_balances (__main__.BankAccountTests.test_04_deposit_withdraw_and_before_after_balances) ... ok
test_05_declined_withdrawal (__main__.BankAccountTests.test_05_declined_withdrawal) ... ok
test_06_shared_interest_and_prompt_correction (__main__.BankAccountTests.test_06_shared_interest_and_prompt_correction) ... ok
test_07_half_even_rounding_and_zero_interest (__main__.BankAccountTests.test_07_half_even_rounding_and_zero_interest) ... ok
test_08_interest_idempotency (__main__.BankAccountTests.test_08_interest_idempotency) ... ok
test_09_interest_idempotency_under_concurrency (__main__.BankAccountTests.test_09_interest_idempotency_under_concurrency) ... ok
test_10_parse_exact_known_timestamp (__


SUCCESS: 25 automated tests passed.


## 5. Design review and production boundaries

**Why a static confirmation parser?** The confirmation holds all of the account identifier, transaction type, UTC timestamp, and transaction ID. The requested local timezone is explicitly supplied. No instance attributes are needed. An arbitrary syntactically valid string can be parsed; `get_transaction()` / `verify_confirmation()` additionally check *this process's* ledger, but do not constitute cryptographic authentication.

**Why both `Decimal` and a currency bound?** Binary floating-point cannot represent all decimal cents exactly; decimal rounding is explicit for calculated interest. The chosen finite balance limit also prevents silently losing cents when arithmetic exceeds the default `decimal` context precision. A real bank would use a currency-specific fixed-scale integer or a carefully configured decimal database column, not this arbitrary teaching bound.

**Interest conventions:** An interest rate is a fractional rate (`0.005` = 0.5%). `pay_interest()` always posts on each call, matching the assignment. `pay_interest("YYYY-MM")` prevents accidental repeat payments **within this object in this process**, returning the original receipt. There is no effective-date rate history or month eligibility rule; neither is specified.

**Transactions and concurrency:** Per-account locks make deposit/withdrawal/interest balance and ledger changes coherent. Bank transfers acquire both locks in a deterministic order and roll back recoverable in-memory errors. `Bank` guarantees account uniqueness inside its registry; standalone accounts or multiple banks do not share a global account registry. The global transaction counter starts at 1 each time the program starts and is not persistent across kernels, processes, or machines.

**Time zones:** Named fixed offsets (`MST`, UTC−07:00) never apply daylight saving. `zoneinfo.ZoneInfo` supports actual location rules when the system's IANA timezone database is available. Transaction timestamps store only whole UTC seconds to match the exercise, so IDs are needed to distinguish events within one second; if exact order for filtered statements across same-second events mattered, retain microseconds in the ledger or use transaction-ID cutoffs.

**Production next steps:** durable database account IDs and atomic journal postings; authorization and ownership checks; encrypted storage, audit trails, and redacted logs; idempotency keys persisted with uniqueness constraints; currency codes and exchange-rate policy; an explicit overdraft and interest-accrual policy; effective-dated rates; double-entry bookkeeping, reversible postings, and reconciliation; appropriate bank-specific limits and compliance review. None of these are provided by this educational notebook.